# Phase 1 — Content-Based Filtering: Cybersecurity Exercise Recommendation System

**COS70008 Technology Innovation Research and Project**  
**Student:** Dana Almadhoun — 106202132  
**Lecturer:** Dr. Saeif Alhazbi | Barzan University College | 2026 SP1

---

## Objective
Build a content-based recommender that answers: *"Given this exercise, which other exercises are most similar to it?"*

**Pipeline:** Raw CSV → Clean & Tag → TF-IDF Vectorisation → Cosine Similarity → Top-5 Recommendations

**Deliverables:**
1. Working Jupyter notebook (this file)
2. Similarity matrix (`ex_sim.npz`)
3. Top-K recommendations CSV (`exercise_item_item_topk.csv`)
4. Heatmap & bar chart visualisations
5. Documentation of design decisions


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.preprocessing import MinMaxScaler
from scipy import sparse
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 80)
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

print("All imports successful.")


## 1. Data Loading & Exploration

Phase 1 uses **only `exercises_full.csv`** (100 exercises × 18 columns). The other files (organisations, ratings) are used in Phase 2.


In [ ]:
# Load all datasets
exercises = pd.read_csv('exercises_full.csv')
orgs = pd.read_csv('orgs_full.csv')
ratings_train = pd.read_csv('ratings_train_full.csv')
ratings_val = pd.read_csv('ratings_validation_full.csv')
ratings_test = pd.read_csv('ratings_test_full.csv')

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
for name, df in [('exercises_full', exercises), ('orgs_full', orgs),
                  ('ratings_train', ratings_train), ('ratings_val', ratings_val),
                  ('ratings_test', ratings_test)]:
    print(f"  {name:25s} → {df.shape[0]:>5} rows × {df.shape[1]:>2} cols")
print(f"  {'Total ratings':25s} → {len(ratings_train)+len(ratings_val)+len(ratings_test):>5}")


In [ ]:
# Explore the exercises dataset — our primary data for Phase 1
print("Columns and types:\n")
print(exercises.dtypes.to_string())
print(f"\nMissing values: {exercises.isnull().sum().sum()} (none)")
exercises.head(3)


In [ ]:
# Identify text (tag-based) columns vs numeric columns
TEXT_COLS = ['ExThreat', 'ExTTPs', 'ExCategories', 'ExGroups', 'ExSoftware',
             'ExStructure', 'ExAudience', 'ExTechniqueIDs', 'ExTactics', 'ExPlatforms']

NUMERIC_COLS = ['ExMaturity', 'ExComplexity', 'ExLength', 'ExTradeCraftIntra', 'ExTradeCraftInter']

# Count unique tags per column
print(f"{'Column':<20} {'Unique Tags':>12}  {'Example values'}")
print("-" * 90)
for col in TEXT_COLS:
    all_tags = set()
    for val in exercises[col]:
        tags = [t.strip() for t in str(val).split(';') if t.strip()]
        all_tags.update(tags)
    example = '; '.join(list(all_tags)[:3])
    print(f"{col:<20} {len(all_tags):>12}  {example}")

print(f"\nTotal unique tags across all text columns will form our feature space.")


In [ ]:
# Numeric column distributions
print("Numeric Column Statistics:\n")
print(exercises[NUMERIC_COLS].describe().round(2).to_string())


In [ ]:
# Visualise numeric distributions
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for i, col in enumerate(NUMERIC_COLS):
    axes[i].hist(exercises[col], bins=15, color='#c0392b', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontsize=11)
    axes[i].set_ylabel('Count' if i == 0 else '')
fig.suptitle('Distribution of Numeric Exercise Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('numeric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


## 2. Data Cleaning & Feature Engineering

### Design Decision: Which Columns to Use

For the content-based recommender, I selected the following **text columns** that describe *what* each exercise covers:

| Column | Why included | Unique tags |
|--------|-------------|-------------|
| `ExThreat` | Core threat type (Ransomware, Phishing, etc.) | 10 |
| `ExTTPs` | ATT&CK tactics and techniques — rich semantic signal | 250 |
| `ExCategories` | General topic categories | 15 |
| `ExGroups` | APT groups — distinctive signal | 155 |
| `ExSoftware` | Malware/tools used — highly specific | 244 |
| `ExTechniqueIDs` | MITRE technique codes (T1059, etc.) — precise identifiers | 303 |
| `ExTactics` | MITRE tactic categories | 14 |
| `ExPlatforms` | Target platforms (Windows, Linux, etc.) | 11 |
| `ExStructure` | Incident response phases covered | 10 |
| `ExAudience` | Target audience | 7 |

**Excluded:** `ExDataSources` — contains long detection descriptions (sentences, not tags), which would introduce noise into a tag-based TF-IDF model. Also excluded `ExCreation` (date, not content).

**Numeric columns** (`ExMaturity`, `ExComplexity`, `ExLength`, `ExTradeCraftIntra`, `ExTradeCraftInter`) are optionally appended after MinMax scaling.


In [ ]:
# ─── Step 1: Clean & Tag ───
# Split semicolon-separated tags, lowercase, strip whitespace, add column prefixes

FEATURE_TEXT_COLS = ['ExThreat', 'ExTTPs', 'ExCategories', 'ExGroups', 'ExSoftware',
                     'ExTechniqueIDs', 'ExTactics', 'ExPlatforms', 'ExStructure', 'ExAudience']

def clean_and_tag(row, columns):
    """
    For each text column, split by semicolon, clean, and prefix with column name.
    Prefixing prevents collisions — e.g., 'Execution' as a tactic vs TTP.
    Returns a single string of space-separated prefixed tokens.
    """
    tokens = []
    for col in columns:
        val = str(row[col]).strip()
        if val and val.lower() != 'nan':
            tags = [t.strip() for t in val.split(';') if t.strip()]
            # Prefix each tag with column name, replace spaces/dots with underscores
            for tag in tags:
                clean_tag = tag.lower().replace(' ', '_').replace('.', '_').replace('-', '_')
                prefix = col.lower().replace('ex', '')  # e.g., ExThreat → threat
                tokens.append(f"{prefix}:{clean_tag}")
    return ' '.join(tokens)

# Apply to every exercise
exercises['combined_tags'] = exercises.apply(
    lambda row: clean_and_tag(row, FEATURE_TEXT_COLS), axis=1
)

# Show example
print("Example — Exercise 1 (first 200 chars):")
print(exercises.loc[0, 'combined_tags'][:200], "...")
print(f"\nTotal tokens in Exercise 1: {len(exercises.loc[0, 'combined_tags'].split())}")
print(f"Average tokens per exercise: {exercises['combined_tags'].apply(lambda x: len(x.split())).mean():.1f}")


## 3. TF-IDF Vectorisation

### Why TF-IDF over One-Hot (Binary) Encoding?

**One-hot encoding** treats every tag equally: `defense-evasion` (appears in ~85/100 exercises) gets the same weight as `access_token_manipulation` (appears in ~3/100). This means common, uninformative tags dominate the similarity calculation.

**TF-IDF** automatically downweights common tags and upweights rare, distinctive ones:
- **TF (Term Frequency):** What proportion of *this* exercise's tags is this particular tag?
- **IDF (Inverse Document Frequency):** How rare is this tag across *all* exercises? `IDF = log(N / df)`

The result: sharing a rare tag like a specific MITRE technique ID contributes far more to similarity than sharing a ubiquitous tactic like `defense-evasion`.


In [ ]:
# ─── Step 2: TF-IDF Vectorisation ───
# Each exercise's combined_tags string is treated as a "document"
# The TfidfVectorizer tokenises on spaces (our tags are already single tokens)

tfidf = TfidfVectorizer(
    analyzer='word',        # split on whitespace
    token_pattern=r'\S+',   # any non-whitespace sequence is a token
    norm='l2',              # L2-normalise each row (standard for cosine sim)
    use_idf=True,           # enable IDF weighting
    smooth_idf=True,        # add 1 to denominator to prevent division by zero
    sublinear_tf=False      # use raw TF (each tag appears at most once per exercise)
)

tfidf_matrix = tfidf.fit_transform(exercises['combined_tags'])

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")
print(f"  → {tfidf_matrix.shape[0]} exercises × {tfidf_matrix.shape[1]} unique features (tags)")
print(f"  → Sparsity: {1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]):.1%}")
print(f"  → Non-zero entries: {tfidf_matrix.nnz}")


In [ ]:
# Inspect: which tags got the highest and lowest IDF weights?
feature_names = tfidf.get_feature_names_out()
idf_scores = tfidf.idf_

idf_df = pd.DataFrame({'tag': feature_names, 'idf': idf_scores}).sort_values('idf')

print("LOWEST IDF (most common, least distinctive):")
print(idf_df.head(10).to_string(index=False))
print("\nHIGHEST IDF (rarest, most distinctive):")
print(idf_df.tail(10).to_string(index=False))


### Optional: Appending Numeric Features

The numeric columns (`ExMaturity`, `ExComplexity`, `ExLength`, etc.) can be appended to the TF-IDF vectors after MinMax scaling to [0, 1]. This lets the system also consider exercise difficulty and duration when computing similarity.

Below I build **two versions** to compare:
1. **Text-only** (TF-IDF on tags)
2. **Text + Numeric** (TF-IDF tags + scaled numeric features)


In [ ]:
# ─── Optional: Append MinMax-scaled numeric features ───
scaler = MinMaxScaler()
numeric_scaled = scaler.fit_transform(exercises[NUMERIC_COLS])
numeric_scaled_df = pd.DataFrame(numeric_scaled, columns=NUMERIC_COLS)

print("Scaled numeric features (first 5 rows):")
print(numeric_scaled_df.head().round(3).to_string())

# Combine: TF-IDF sparse matrix + dense numeric columns
from scipy.sparse import hstack
tfidf_plus_numeric = hstack([tfidf_matrix, sparse.csr_matrix(numeric_scaled)])

print(f"\nText-only matrix:      {tfidf_matrix.shape}")
print(f"Text + Numeric matrix: {tfidf_plus_numeric.shape}")


## 4. Cosine Similarity

### Why Cosine Similarity?

Cosine similarity measures the **angle** between two vectors, not their magnitude. This is ideal because:

- An exercise with 3 tags and one with 15 tags can still be compared fairly — it only cares about the *pattern* of tags, not the count.
- It naturally pairs with TF-IDF (which produces L2-normalised vectors).
- Score range is [0, 1]: 0 = nothing in common, 1 = identical.

**Formula:** `cos(A, B) = (A · B) / (||A|| × ||B||)`

**Alternatives explored:**
- **Jaccard similarity:** Works on binary sets (shared tags / all unique tags). Simpler but cannot leverage TF-IDF weights.
- **Euclidean distance:** Penalises vectors of different lengths; needs inversion for similarity.


In [ ]:
# ─── Step 3: Compute Pairwise Cosine Similarity ───

# Primary: Text-only TF-IDF
cosine_sim_text = cosine_similarity(tfidf_matrix)

# Alternative: Text + Numeric
cosine_sim_combined = cosine_similarity(tfidf_plus_numeric)

print(f"Cosine Similarity Matrix (Text-only):   {cosine_sim_text.shape}")
print(f"Cosine Similarity Matrix (Text+Numeric): {cosine_sim_combined.shape}")
print(f"\nDiagonal check (should be 1.0): {cosine_sim_text[0, 0]:.4f}")
print(f"Similarity between Ex1 & Ex2:   {cosine_sim_text[0, 1]:.4f}")
print(f"Similarity between Ex1 & Ex3:   {cosine_sim_text[0, 2]:.4f}")


In [ ]:
# ─── Alternative: Jaccard Similarity (Binary Encoding) for comparison ───

def build_binary_matrix(df, columns):
    """Build a binary (one-hot) tag matrix for Jaccard comparison."""
    all_tags_per_row = []
    for _, row in df.iterrows():
        tags = set()
        for col in columns:
            val = str(row[col]).strip()
            if val and val.lower() != 'nan':
                for t in val.split(';'):
                    clean = t.strip().lower().replace(' ', '_')
                    tags.add(f"{col.lower().replace('ex','')}:{clean}")
        all_tags_per_row.append(tags)
    
    all_unique = sorted(set.union(*all_tags_per_row))
    binary = np.zeros((len(df), len(all_unique)))
    for i, tags in enumerate(all_tags_per_row):
        for j, tag in enumerate(all_unique):
            if tag in tags:
                binary[i, j] = 1
    return binary

binary_matrix = build_binary_matrix(exercises, FEATURE_TEXT_COLS)

# Jaccard: |A ∩ B| / |A ∪ B|
def jaccard_similarity_matrix(mat):
    n = mat.shape[0]
    sim = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            intersection = np.sum((mat[i] == 1) & (mat[j] == 1))
            union = np.sum((mat[i] == 1) | (mat[j] == 1))
            score = intersection / union if union > 0 else 0
            sim[i, j] = score
            sim[j, i] = score
    return sim

jaccard_sim = jaccard_similarity_matrix(binary_matrix)

print(f"Jaccard Similarity Matrix: {jaccard_sim.shape}")
print(f"Jaccard — Ex1 vs Ex2: {jaccard_sim[0, 1]:.4f}")
print(f"Jaccard — Ex1 vs Ex3: {jaccard_sim[0, 2]:.4f}")


In [ ]:
# ─── Compare: Cosine (TF-IDF) vs Jaccard (Binary) vs Cosine (Text+Numeric) ───

# Extract upper triangle values (excluding diagonal) for distribution comparison
def upper_tri_values(mat):
    return mat[np.triu_indices_from(mat, k=1)]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, sim, title, color in zip(axes,
    [cosine_sim_text, jaccard_sim, cosine_sim_combined],
    ['Cosine (TF-IDF Text)', 'Jaccard (Binary)', 'Cosine (Text + Numeric)'],
    ['#c0392b', '#2980b9', '#27ae60']):
    vals = upper_tri_values(sim)
    ax.hist(vals, bins=50, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Similarity Score')
    ax.set_ylabel('Pair Count')
    ax.axvline(vals.mean(), color='black', linestyle='--', linewidth=1.5, label=f'Mean: {vals.mean():.3f}')
    ax.legend()

plt.suptitle('Distribution of Pairwise Similarity Scores Across All Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('similarity_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Top-K Recommendations

For each exercise, sort all other exercises by cosine similarity (descending) and select the **top 5** most similar. This is the final recommendation output.


In [ ]:
# ─── Step 4: Generate Top-5 Recommendations ───
# Using the PRIMARY method: Cosine similarity on TF-IDF text features

K = 5  # Top-K neighbours

def get_top_k(sim_matrix, exercise_ids, k=5):
    """For each exercise, return the top-k most similar exercises (excluding itself)."""
    records = []
    for i in range(sim_matrix.shape[0]):
        # Get similarity scores for exercise i, excluding itself
        scores = sim_matrix[i].copy()
        scores[i] = -1  # exclude self
        
        # Get top-k indices
        top_indices = np.argsort(scores)[::-1][:k]
        
        for rank, j in enumerate(top_indices, 1):
            records.append({
                'EXID': exercise_ids[i],
                'Rank': rank,
                'Similar_EXID': exercise_ids[j],
                'Similarity': round(scores[j], 4),
                'ExThreat_Source': exercises.loc[i, 'ExThreat'],
                'ExThreat_Similar': exercises.loc[j, 'ExThreat']
            })
    return pd.DataFrame(records)

exercise_ids = exercises['EXID'].values
topk_df = get_top_k(cosine_sim_text, exercise_ids, k=K)

print(f"Generated {len(topk_df)} recommendations ({len(exercises)} exercises × {K} each)")
print(f"\nTop-5 recommendations for Exercise 1:")
print(topk_df[topk_df['EXID'] == 1].to_string(index=False))


In [ ]:
# Show recommendations for a few diverse exercises
for ex_id in [1, 25, 50, 75, 100]:
    threat = exercises.loc[exercises['EXID'] == ex_id, 'ExThreat'].values[0]
    print(f"\nExercise {ex_id} ({threat}):")
    subset = topk_df[topk_df['EXID'] == ex_id][['Rank', 'Similar_EXID', 'Similarity', 'ExThreat_Similar']]
    print(subset.to_string(index=False))


## 6. Visualisations

### 6.1 Similarity Heatmap (100 × 100)


In [ ]:
# ─── Step 5a: Heatmap of the full 100×100 similarity matrix ───
fig, ax = plt.subplots(figsize=(14, 12))

sns.heatmap(cosine_sim_text, 
            cmap='YlOrRd',
            xticklabels=exercises['EXID'].values,
            yticklabels=exercises['EXID'].values,
            vmin=0, vmax=1,
            linewidths=0,
            ax=ax)

ax.set_title('Exercise-Exercise Cosine Similarity Heatmap (TF-IDF)', fontsize=16, fontweight='bold')
ax.set_xlabel('Exercise ID', fontsize=12)
ax.set_ylabel('Exercise ID', fontsize=12)

# Reduce tick label clutter
ax.set_xticks(range(0, 100, 5))
ax.set_xticklabels(range(1, 101, 5), fontsize=7, rotation=90)
ax.set_yticks(range(0, 100, 5))
ax.set_yticklabels(range(1, 101, 5), fontsize=7)

plt.tight_layout()
plt.savefig('heatmap_similarity.png', dpi=150, bbox_inches='tight')
plt.show()

print("Heatmap saved to heatmap_similarity.png")


### 6.2 Bar Charts — Top-5 Similar Exercises


In [ ]:
# ─── Step 5b: Bar charts for selected exercises ───
sample_exercises = [1, 25, 50, 75]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for idx, ex_id in enumerate(sample_exercises):
    ax = axes[idx]
    subset = topk_df[topk_df['EXID'] == ex_id].sort_values('Rank')
    
    bars = ax.barh(
        [f"Ex {r['Similar_EXID']} ({r['ExThreat_Similar'][:15]})" for _, r in subset.iterrows()],
        subset['Similarity'],
        color=['#c0392b', '#e74c3c', '#e67e22', '#f39c12', '#f1c40f'],
        edgecolor='white'
    )
    
    # Add value labels
    for bar, val in zip(bars, subset['Similarity']):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=10)
    
    threat = exercises.loc[exercises['EXID'] == ex_id, 'ExThreat'].values[0]
    ax.set_title(f'Exercise {ex_id} — {threat}', fontsize=12, fontweight='bold')
    ax.set_xlim(0, 1.0)
    ax.set_xlabel('Cosine Similarity')
    ax.invert_yaxis()

plt.suptitle('Top-5 Most Similar Exercises (Content-Based, TF-IDF + Cosine)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('top5_bar_charts.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ─── Additional: Average similarity by threat type ───
threats = exercises['ExThreat'].apply(lambda x: x.split(';')[0].strip())
unique_threats = sorted(threats.unique())

threat_sim = np.zeros((len(unique_threats), len(unique_threats)))
threat_counts = np.zeros((len(unique_threats), len(unique_threats)))

for i in range(100):
    for j in range(100):
        ti = unique_threats.index(threats.iloc[i])
        tj = unique_threats.index(threats.iloc[j])
        threat_sim[ti, tj] += cosine_sim_text[i, j]
        threat_counts[ti, tj] += 1

threat_sim_avg = np.divide(threat_sim, threat_counts, where=threat_counts > 0)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(threat_sim_avg, annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=unique_threats, yticklabels=unique_threats, ax=ax)
ax.set_title('Average Similarity by Threat Type', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('threat_similarity.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Export Deliverables


In [ ]:
# ─── Save similarity matrix as .npz (sparse format for Phase 2 / dashboard) ───
sparse_sim = sparse.csr_matrix(cosine_sim_text)
sparse.save_npz('ex_sim.npz', sparse_sim)
print(f"✓ Saved ex_sim.npz ({sparse_sim.shape})")

# ─── Save Top-K recommendations CSV ───
topk_export = topk_df[['EXID', 'Rank', 'Similar_EXID', 'Similarity']].copy()
topk_export.to_csv('exercise_item_item_topk.csv', index=False)
print(f"✓ Saved exercise_item_item_topk.csv ({len(topk_export)} rows)")

# ─── Also save the full similarity matrix as CSV (human-readable) ───
sim_df = pd.DataFrame(cosine_sim_text,
                       index=exercises['EXID'],
                       columns=exercises['EXID']).round(4)
sim_df.to_csv('similarity_matrix_full.csv')
print(f"✓ Saved similarity_matrix_full.csv (100 × 100)")

# Quick verification
loaded = sparse.load_npz('ex_sim.npz')
print(f"\nVerification — loaded matrix shape: {loaded.shape}, max: {loaded.max():.4f}")


## 8. Evaluation & Quality Checks

Since Phase 1 is unsupervised (no ground-truth labels for "correct" recommendations), we evaluate using:

1. **Similarity score distribution** — Are recommendations meaningfully similar (not too low, not all 1.0)?
2. **Coverage** — Does every exercise have reasonable recommendations?
3. **Sanity checks** — Do exercises about the same threat type get recommended together?
4. **Comparison across metrics** — Does TF-IDF + Cosine outperform Jaccard?


In [ ]:
# ─── Evaluation Metrics ───

# 1. Score distribution of top-5 recommendations
top5_scores = topk_df['Similarity']
print("Top-5 Recommendation Score Statistics:")
print(f"  Mean similarity:   {top5_scores.mean():.4f}")
print(f"  Median:            {top5_scores.median():.4f}")
print(f"  Min:               {top5_scores.min():.4f}")
print(f"  Max:               {top5_scores.max():.4f}")
print(f"  Std:               {top5_scores.std():.4f}")

# 2. Coverage: every exercise has 5 recommendations
assert topk_df.groupby('EXID').size().nunique() == 1, "Not all exercises have K recs!"
print(f"\n✓ Coverage: All {topk_df['EXID'].nunique()} exercises have exactly {K} recommendations.")

# 3. Sanity check: same-threat hit rate
# For each exercise, how many of its top-5 share the same primary threat?
exercises['PrimaryThreat'] = exercises['ExThreat'].apply(lambda x: x.split(';')[0].strip())
threat_map = dict(zip(exercises['EXID'], exercises['PrimaryThreat']))

hits = 0
total = 0
for _, row in topk_df.iterrows():
    if threat_map[row['EXID']] == threat_map[row['Similar_EXID']]:
        hits += 1
    total += 1

print(f"\nSame-threat hit rate: {hits}/{total} = {hits/total:.1%}")
print(f"  (How often a recommended exercise shares the same primary threat type)")


In [ ]:
# ─── Compare: TF-IDF+Cosine vs Jaccard ───

# Generate Jaccard top-5 for comparison
topk_jaccard = get_top_k(jaccard_sim, exercise_ids, k=K)

# Same-threat hit rate for Jaccard
hits_j = sum(1 for _, r in topk_jaccard.iterrows()
             if threat_map[r['EXID']] == threat_map[r['Similar_EXID']])

print("Method Comparison — Same-Threat Hit Rate:")
print(f"  TF-IDF + Cosine:  {hits/total:.1%}")
print(f"  Binary + Jaccard: {hits_j/len(topk_jaccard):.1%}")

# Overlap: how many recommendations are the same between the two methods?
cosine_recs = set(zip(topk_df['EXID'], topk_df['Similar_EXID']))
jaccard_recs = set(zip(topk_jaccard['EXID'], topk_jaccard['Similar_EXID']))
overlap = len(cosine_recs & jaccard_recs)
print(f"\nRecommendation overlap: {overlap}/{len(cosine_recs)} = {overlap/len(cosine_recs):.1%}")
print(f"  (How many of the top-5 recommendations are identical between the two methods)")


## 9. Design Decision Summary

### Feature Selection
I used **10 text columns** covering threats, TTPs, technique IDs, APT groups, software, tactics, platforms, audience, structure, and categories. I excluded `ExDataSources` because its values are full sentences rather than clean tags — including them would introduce noise. I excluded `ExCreation` (a date with no content meaning).

### Vectorisation: TF-IDF
**TF-IDF** was chosen over binary (one-hot) encoding because:
- It automatically distinguishes rare, informative tags from ubiquitous ones
- A tag like `defense-evasion` (in 85% of exercises) gets near-zero weight, while `access_token_manipulation` (in 3%) gets high weight
- This reflects real-world intuition: sharing a rare technique is a much stronger signal of similarity

### Similarity Metric: Cosine Similarity
**Cosine similarity** was chosen because:
- It measures direction (pattern of tags), not magnitude — fair comparison between exercises with different numbers of tags
- Naturally bounded [0, 1], interpretable as "percentage of alignment"
- Standard pairing with TF-IDF vectors

### Prefix Tagging
Each tag is prefixed with its source column (e.g., `threat:ransomware`, `tactics:defense_evasion`). This prevents collisions where the same word means different things in different columns (e.g., "Execution" as a TTP vs as a tactic).

### Output Files
| File | Description |
|------|-------------|
| `ex_sim.npz` | 100×100 sparse cosine similarity matrix |
| `exercise_item_item_topk.csv` | Top-5 recommendations per exercise |
| `similarity_matrix_full.csv` | Full matrix in human-readable CSV |
| `heatmap_similarity.png` | Visual heatmap of all pairwise similarities |
| `top5_bar_charts.png` | Bar charts of top-5 for selected exercises |
